In [ ]:
from datetime import datetime
from pathlib import Path
import time
import re
import win32com.client
import random

# parameters
VL_SUBJECT = "[Availability][Personal Time-Off] "
SL_SUBJECT = "[Availability][Sick Leave] "
PROJECT_BACKUPS = {
    "Argus": [
        "ian.bautista@lingarogroup.com",
    ],
    "CCH": [
        "erin.aguilar@lingarogroup.com",
    ],
}

def insert_body_before_signature(signature_html, body_html):
    """
    Insert the custom email HTML inside Outlook's existing <body> tag.
    """
    if re.search(r"<body[^>]*>", signature_html, re.IGNORECASE):
        return re.sub(
            r"(<body[^>]*>)",
            lambda match: match.group(1) + body_html,
            signature_html,
            count=1,
            flags=re.IGNORECASE
        )

    return body_html + signature_html

def build_project_backup_rows():
    """
    Builds the Project Engagement / Backup table rows.
    """
    rows = ""
    for project, backups in PROJECT_BACKUPS.items():
        backup_list = "<br>".join(backups)

        rows += f"""
        <tr>
            <td><b>{project}</b></td>
            <td>{backup_list}</td>
        </tr>
        """

    return rows

def checkDateInput(dateStart, dateEnd):
    try: 
        if dateEnd < dateStart:
            print("Start Date cannot be later than End Date")
            return False
    
        return True;
    except ValueError:
        print("Invalid date format. Please use MM-DD-YYYY.")
        return False

def generate_sick_leave_message():
    sl_body = {
        "messages": [
            """<p>
            I’m feeling unwell and will be taking sick leave. I’ll rest and monitor my condition, and I’ll keep you updated if I need additional time off.
            </p>""",
            """<p>
            I’m currently unwell and need to take sick leave. I’ll focus on recovering and will keep you informed if there are any changes to my expected return date. Thank you for your consideration.
            </p>""",
            """<p>
            I’m not feeling well and would like to take sick leave. I’ve informed the team about any urgent tasks and will respond to important messages when I’m able to. I’ll provide an update regarding my return to work as soon as possible.
            </p>
            """
        ]
    }

    message = sl_body["messages"][random.randint(0, (len(sl_body["messages"]) - 1))]

    return message

def generateEmailVL(start, end, purpose):
    """
    Generates an Outlook email with your custom body and signature.

    start/end should be datetime objects.
    purpose should be "VL" or "SL".
    """
    leave_type = "Personal Leave"
    formatted_start = start.strftime("%B %d, %Y")
    formatted_end = end.strftime("%B %d, %Y")
    subject = f"{VL_SUBJECT}{formatted_start} - {formatted_end}"
    # Generate the project + backup rows from PROJECT_BACKUPS.
    project_backup_rows = build_project_backup_rows()
    body_html = f"""
        <div style="font-family: Calibri, Arial, sans-serif; font-size: 11pt;">
            <p>Hello, SLMs</p>
    
            <p>
                Asking for your approval on my time-off request below.
                This has been aligned to my indicated backup in each project.
                Thank you!
            </p>
    
            <table border="1" cellspacing="0" cellpadding="6"
                   style="border-collapse: collapse;
                          font-family: Calibri, Arial, sans-serif;
                          font-size: 11pt;">
                <tr style="background-color: #F2F2F2;">
                    <th colspan="2">Leave Request Checklist</th>
                </tr>
    
                <tr>
                    <td><b>Type of Leave</b></td>
                    <td>{leave_type}</td>
                </tr>
    
                <tr>
                    <td><b>Requested Leave Dates</b></td>
                    <td>{formatted_start} to {formatted_end}</td>
                </tr>
    
                <tr>
                    <td><b>Sprout and SULU Request Screenshot</b></td>
                    <td>&nbsp;</td>
                </tr>
    
                <tr style="background-color: #F2F2F2;">
                    <th>Project Engagement</th>
                    <th>Backup</th>
                </tr>
    
                {project_backup_rows}
            </table>
    
            <p>Please let me know if you need any additional information.</p>
    
            <p>Thank you.</p>
            <br>
        </div>
        """

    # Generate outlook email
    try:
        outlook = win32com.client.Dispatch("Outlook.Application")
        email = outlook.CreateItem(0)
        email.BodyFormat = 2
        email.To = "; ".join(RECIPIENTS)
        email.Subject = subject
        email.Display(False)
        time.sleep(2)
        signature_html = email.HTMLBody
        email.HTMLBody = insert_body_before_signature(signature_html, body_html)
        email.Display(False)
        print("Outlook email succesfully created")

    except Exception as error:
        print(f"Something went wrong with generating the email: {error}")


    # Save generate outlook email to output path
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
    filename = (f"{purpose}_{start.strftime('%m-%d-%Y')}_to_{end.strftime('%m-%d-%Y')}_{timestamp}.msg")
    file_path = OUTPUT_PATH / filename
    email.SaveAs(str(file_path), 3)
    print(f"Saved email to: {file_path}")

    return email

def generateEmailSL(start, end, purpose):
    """
    Generates an Outlook email with custom body and signature.

    start/end should be datetime objects.
    purpose should be "VL" or "SL"
    """
    leave_type = "Sick Leave"
    formatted_start = start.strftime("%B %d, %Y")
    formatted_end = end.strftime("%B %d, %Y")
    subject = f"{VL_SUBJECT}{formatted_start} - {formatted_end}"
    sl_message = generate_sick_leave_message()

    body = f"""
        <div style="font-family: Calibri, Arial, sans-serif; font-size: 11pt;">
            <p>Hello, SLMs</p>
    
            {sl_message}
    
            <p>Thank you.</p>
            <br>
        </div>
        """

    

    print(body)

def main():
    dateValidation = True
    # while dateValidation:
    #     dateStart = datetime.strptime(input("Date Start (MM-DD-YYYY): ").strip(), "%m-%d-%Y")
    #     ateEnd = datetime.strptime(input("Date End (MM-DD-YYYY): ").strip(), "%m-%d-%Y")
    #     res = checkDateInput(dateStart, dateEnd)
    #     if res == True:
    #         dateValidation = False
    
    purpose = ""
    while purpose not in ["VL", "SL"]:
        print("Invalid purpose. Enter VL or SL only.")
        purpose = input("Purpose? (VL or SL): ").strip().upper()

    if purpose == "VL":
        try:
            generateEmailVL(dateStart, dateEnd, purpose)
        except Exception as error:
            print(f"Something whet wrong with calling generateEmailVL(): {error}")

    if purpose == "SL":
        try:
            generateEmailSL(dateStart, dateEnd, purpose)
        except Exception as error:
            print(f"Something wrong happened executing generateEmailSL(): {error}") 

purpose = "SL"
dateStart = datetime.strptime("12-12-2026", "%m-%d-%Y")
dateEnd = datetime.strptime("12-15-2026", "%m-%d-%Y")

if __name__ == "__main__":
    main()

Invalid purpose. Enter VL or SL only.



        <div style="font-family: Calibri, Arial, sans-serif; font-size: 11pt;">
            <p>Hello, SLMs</p>

            <p>
            I’m not feeling well and would like to take sick leave. I’ve informed the team about any urgent tasks and will respond to important messages when I’m able to. I’ll provide an update regarding my return to work as soon as possible.
            </p>
            

            <p>Please let me know if you need any additional information.</p>

            <p>Thank you.</p>
            <br>
        </div>
        
